# Read-only audit: CUDA/T4 V-14 2x core artifacts

Run this only after the recovered `b0000`--`b0236` directories and the completed `b0237`--`b0333` directories are co-located in one Drive artifact root. It validates all 334 sealed batch artifacts against the exact accepted 2x plan.

It reads only. It writes no merged index and prints only non-scientific provenance: plan hash, counts, and a receipt hash. It does not expose or calculate values, aggregates, rates, comparisons, or conclusions.

In [ ]:
from google.colab import files, drive
from pathlib import Path
import hashlib
import json
import sys
import tarfile

sys.dont_write_bytecode = True
EXPECTED_CAPSULE_SHA256 = 'ef0681f50165615cf9a862778f80ce65b58b2b643a7fe4d2612a16dc59b71337'
uploaded = files.upload()  # Select g6_cuda_t4_v14_core_final_2x_capsule.tar.gz
assert len(uploaded) == 1
archive = Path(next(iter(uploaded)))
assert hashlib.sha256(archive.read_bytes()).hexdigest() == EXPECTED_CAPSULE_SHA256
with tarfile.open(archive, 'r:gz') as bundle:
    bundle.extractall('/content', filter='data')
ROOT = Path('/content/project')
CONTROL_INPUT = Path('/content/g6_v14_2x_core_control')
assert (ROOT / 'research/g6/cuda_qualification/v14_core_batch_contract.py').is_file() and CONTROL_INPUT.is_dir()
drive.mount('/content/gdrive')
print('Read-only audit environment ready.')

In [ ]:
# Set this only after both independently sealed 2x directory sets are co-located here.
ARTIFACT_ROOT = Path('/content/gdrive/MyDrive/g6_cuda_t4_v14_core_2x_reconciled_20260921/artifacts')
EXPECTED_PLAN_CONTENT_SHA256 = '38c01eff5a313273fc106a5f5c39760628e59132b9a7a72e0176b37aa1897748'

sys.path.insert(0, str(ROOT))
from research.g6.cuda_qualification import v14_core_batch_contract as contract

plan_path = CONTROL_INPUT / 'v14_2x_core_batch_plan.json'
assert plan_path.is_file() and ARTIFACT_ROOT.is_dir(), 'Plan or reconciled artifact root is missing.'
plan = json.loads(plan_path.read_text(encoding='utf-8'))
assert plan['content_sha256'] == EXPECTED_PLAN_CONTENT_SHA256 and plan['scale'] == '2x'
contract.validate_plan(plan)

# Purely read-only: the complete artifact receipt remains in memory and is not written to Drive.
receipt = contract.validate_final_set(plan, ARTIFACT_ROOT)
assert receipt['verdict'] == 'accepted'
assert receipt['batch_count'] == 334 and receipt['total_coordinates'] == 1367631
print('2x artifact audit: PASS')
print('plan content SHA-256:', receipt['plan_content_sha256'])
print('artifact batches:', receipt['batch_count'])
print('coordinates:', receipt['total_coordinates'])
print('read-only receipt content SHA-256:', receipt['content_sha256'])